# **CMSC 320 - FINAL PROJECT**

In [ ]:
#imports
import pandas as pd
import string
import math
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd




**PART 1 - DATA PROCESSING**

We are using Earthquake data from the USGS earthquake data website.  This data is from June 1st, 2005 to September 27th, 2012. It includes not only earthquake data but data involving Volcano Eruption and Landsides all of which has been reviewd in the context of Earthquakes.

In [ ]:
earthquakes_df = pd.read_csv("Earthquakes.csv")
earthquakes_df

Information about the dataset before any modifications take place

In [57]:
print(f"df.shape: {earthquakes_df.shape}")
earthquakes_df.shape
print("df.describe(): ")
display(earthquakes_df.describe())
print("df.info(): ")
earthquakes_df.info()



df.shape: (16000, 23)
df.describe(): 


,latitude,longitude,depth,mag,nst,gap,dmin,rms,horizontalError,depthError,magError,magNst
count,16000.000000,16000.000000,16000.000000,16000.00000,14720.000000,14717.000000,1490.000000,7664.000000,1513.000000,4770.000000,1176.000000,6173.000000
mean,27.611573,-104.703152,38.187911,4.10787,47.583628,168.173160,0.278961,0.850562,1.483323,14.140468,0.191622,51.673092
std,16.763399,31.873068,43.143381,0.49540,77.578092,79.883874,0.357982,0.889954,5.029209,155.091728,0.157575,84.415456
min,7.959000,-171.562000,-2.285000,3.50000,0.000000,11.900000,0.001802,0.009800,0.080000,0.000000,0.000000,0.000000
25%,15.886000,-120.798542,10.000000,3.70000,8.000000,107.000000,0.054320,0.570000,0.300000,0.500000,0.134000,5.000000
50%,18.629000,-98.362000,19.000000,4.00000,18.000000,170.200000,0.166600,0.920000,0.580000,4.025000,0.163000,15.000000
75%,40.308958,-87.848000,51.700000,4.40000,49.000000,219.800000,0.392775,1.150000,1.630000,12.000000,0.200000,50.000000
max,72.852000,-36.609000,294.900000,7.60000,857.000000,358.300000,5.654000,69.320000,99.000000,9117.600000,1.680000,438.000000


df.info(): 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             16000 non-null  object 
 1   time             16000 non-null  object 
 2   latitude         16000 non-null  float64
 3   longitude        16000 non-null  float64
 4   depth            16000 non-null  float64
 5   mag              16000 non-null  float64
 6   magType          16000 non-null  object 
 7   nst              14720 non-null  float64
 8   gap              14717 non-null  float64
 9   dmin             1490 non-null   float64
 10  rms              7664 non-null   float64
 11  net              16000 non-null  object 
 12  id               16000 non-null  object 
 13  updated          16000 non-null  object 
 14  place            16000 non-null  object 
 15  type             16000 non-null  object 
 16  horizontalError  1513 non-null   float64
 17  

Parse data: Sort by magnitude

In [ ]:
#sort by mag highest to smallest
earthquakes_df = earthquakes_df.sort_values(by=['mag'], ascending=False).reset_index(drop='True')

#this line below is to check if there was mag error, there is, maybe we could use it
#earthquakes_df = earthquakes_df.dropna(subset=['magError'])

#date only column, in datetime (remove .dt.date to get full date and time in datetime format)
date = pd.to_datetime(earthquakes_df['time']).dt.date
earthquakes_df.insert(0, 'date', date)
display(earthquakes_df)

**PART 2**

Method 1: Frequency of Earthquakes Based on Geographical Region using Chi Square Test 


We will first examine the frequency of earthquakes based off of geographical region. To better visualize the correlation between how often the earthquakes happen and which locations get the most number of earthquakes, we will use a bar chart and a pie chart. First, we want to see how frequent are the earthquakes in the locations in our database. We notice that most of the locations have only 1 earthquake, with the graph being extremely right skewed that is almost imposible to be seen on the graph. From this first graph we understand that most of the locations in our data base have less than 10 earthquakes.

In [ ]:
place_types = earthquakes_df['place'].value_counts()
#print(place_types.count())

bins = [0,1, 5, 10, 20, float('inf')]
labels = ['1','2-5','6-10', '11-20', '21+']

binned = pd.cut(place_types, bins = bins, labels=labels)
binned_occurences = binned.value_counts().sort_index()

binned_occurences.plot(kind='bar')
plt.title('How often do earthquakes happen in our locations')
plt.xlabel('Number of earthquakes')
plt.ylabel('Number of locations')

plt.show()

We wanted to take a closer look at the locations that have more than 10 earthquakes. We wanted to see exactly which locations are prone to having more earthquakes and their exact number of earthquakes. We notice that there are 2 locations with an astonishing amount of earthquakes, more than 100. These 2 locations are the outliers that are skewing the graph. This made us wonder what is the distribution of the frequency of earthquakes.

In [ ]:
place_types_more_than_10_occurences = place_types[place_types > 10]

place_types_more_than_10_occurences.sort_values().plot(kind='bar')

plt.title("Locations with more than 10 earthquakes")
plt.ylabel('Number of earthquakes')
plt.xlabel('Locations')
plt.show()

Among out dataset, we wondered what is the percentage of locations that suffered 1 earthquake, between 2 and 10, and finally, more than 11. For this question we used the pie chart which perfectly shows that almost 88% of the locations have suffered only 1 earthquake, while 0.174% have suffered more than 11 earthquakes.

In [ ]:
bins = [0,1, 10, float('inf')]
labels = ['1','2-10', '11+']

binned = pd.cut(place_types, bins = bins, labels=labels)
binned_occurences = binned.value_counts().sort_index()


legend=['1 earthquake', '2-10 earthquakes', 'more than 11 earthquakes']
myexplode=[0, 0, 0.7]
binned_occurences.plot(kind='pie', labels=None, autopct='%1.3f%%')
plt.title('Distribution of frequency of earthquakes')
plt.legend(legend)
plt.show()

This brought us to our next question, are certain areas more prone to earthquakes with higher or lower frequency? We want to see the relationship and the ratio between the frequency of the earthquakes and the areas where these earthquakes happened. We will be using the Chi-Square Test for Independence and our Null Hypothesis is that there is no relation.

In [ ]:
#place_types = earthquakes_df['place'].value_counts()

my_places = place_types

my_places = my_places.reset_index()
my_places.columns = ['place', 'count']

#calculate the average latitude for each place
latitude_table = earthquakes_df.groupby('place', as_index=False)['latitude'].mean()

earthquake_dataframe = pd.merge(
    my_places,
    latitude_table,
    on='place',
    how='inner'
)

#this is variable 1
bins = [0, 1, 10, float('inf')]
labels = ['1 earthquake', '2 to 10 earthquakes', 'more than 10 earthquakes']
earthquake_dataframe['count category'] = pd.cut(earthquake_dataframe['count'], bins = bins, labels=labels)

#this is variable 2
bins = [0, 30, 60, 90]
labels = ['Low Latitude', 'Mid Latitude', 'High Latitude']
earthquake_dataframe['region category'] = pd.cut(earthquake_dataframe['latitude'], bins = bins, labels=labels)


#i need contingency table
contingency_table = pd.crosstab(earthquake_dataframe['count category'], earthquake_dataframe['region category'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(contingency_table)

print("\np-value:", p_value)

if p_value <= 0.05:
    print('\nDependent (reject H0)')
else:
    print('\nIndependent (H0 holds true)')

The Chi-square test shows that there is a relation between the lower, middle, higher part of the locations in our dataframe, and the earthquake frequency. From the table we can see that locations with only one earthquake are predominantly in the lower region, and the earthquakes are not evenly distributed across lower, middle and higher regions.

Method 2: Magnitude vs Depth using Pearson Correlation

Earlier we found that the Low Latitude region contained the greatest number of earthquake locations across all frequency categories. We will now use this region to investigate whether earthquake magnitude and depth are correlated.

edit this: Now looking at the data below looks superoverwhelming but thats because location source directly from the USGS center is a lot but is can be seen that over the years USGS has recorded less earthquakes with a magnitude less than 4 and has recorded more earthquakes with a magnitude over 7 past 2011. This shows that over time the strength of the earthquakes have increased. From 2009 you can start to see less earthquakes under 4 with more occurances of earthquakes with mangnitude between 5 and 7.

In [ ]:
south_locations = earthquake_dataframe[earthquake_dataframe['region category'] == 'Low Latitude']['place']

region = earthquakes_df[earthquakes_df['place'].isin(south_locations)].copy()

rvalue = region['mag'].corr(region['depth'])
print(f"Pearson correlation (Magnitude vs. Depth): {rvalue}")

plt.figure(figsize=(8, 10))
plt.scatter(region['mag'], region['depth'])
plt.title("Earthquake Magnitude vs. Depth (Low Latitude Group)")
plt.xlabel("Magnitude")
plt.ylabel("Depth (km)")
plt.show()



There is a weak negative correlation but because there is so much data, we need to break some of this information down. We are going to still look at the south group. But we are going to visualize how many earthquakes there are by date.

In [ ]:
region['date'] = pd.to_datetime(region['date'])

region['date'].value_counts().sort_index().plot()
plt.xlabel("Date")
plt.ylabel("Earthquakes Occured")
plt.title("Earthquakes occured on each day recorded by USGS center")
plt.show()

We can see a huge spikes of earthquakes in around 2008, 2010, and 2013. But we are going to break this down a little further, and see the break down in just 2010 and see if there is more of a corrleation between Depth and magnitude since there will be less data points.

In [ ]:
region['year'] = pd.to_datetime(region['date']).dt.year
south_earthquakes = region.groupby(region['year'])
year = south_earthquakes.get_group(2010)

#then find magnitude v depth
rvalue2010 = year['depth'].corr(year['mag'])
print(f"Pearson r value for 2010 for depth and mag is: {rvalue2010}")
plt.figure(figsize=(8, 10))
plt.scatter(year['mag'], year['depth'])
plt.title("Earthquake Magnitude v. Depth in 2010")
plt.show()

edit: There is still a lot of data points and the R-value is still small but looking at the second graph it can be seen in 2010 that the larger than 5.5 that a negative corrleation to the depth of the earthquake.

**NOTE FOR GROUP MATES:**
what I did is I switched the methods to make it flow a little better. So Ana's analysis was about frequency in each region, and it concluded that the South divide of the globe contained a higher freq of earthquakes. i didn't edit her code. in order to make it flow like a story, i edited Natalie's method to examine the correlation of mag vs depth in that Southern region that had a higher frequency. I'm not sure how effective this is, i just wanted to relate the themes of our methods together. I wonder if in Natalie's method, we should find correlation between mag and depth for all 3 regions instead of zooming in on just one? 

Method 3: Comparison of Earthquake Characteristics Across Latitude Regions using ANOVA testing

We have three divides of the regions: Low Latitude, Mid Latitude, and High Latitude

We know want to examine and compare the average magnitude and depth of the recorded earthquakes between each divide. Earlier we observed that the Low Latitude region contains the largest number of earthquake locations across all frequency categories, followed by the Mid Latitude region, while the High Latitude region has the fewest. Let's perform Anova to examine the average depths across the different regions:

In [ ]:
#divide the dataframe into the regions
groups = earthquake_dataframe.groupby("region category",observed=False)

low = groups.get_group("Low Latitude")
mid = groups.get_group("Mid Latitude")
high = groups.get_group("High Latitude")

#grab the places within each region from the original dataframe
low_places = low["place"]
low_earthquakes = earthquakes_df[earthquakes_df["place"].isin(low_places)]

mid_places = mid["place"]
mid_earthquakes = earthquakes_df[earthquakes_df["place"].isin(mid_places)]

high_places = high["place"]
high_earthquakes = earthquakes_df[earthquakes_df["place"].isin(high_places)]

#print the avg depths
print("Mean depth of lower lat regions:",low_earthquakes['depth'].mean())
print("Mean depth of middle lat regions:", mid_earthquakes['depth'].mean())
print("Mean depth of higher lat regions:", high_earthquakes['depth'].mean())

#perform the ANOVA test
f_stat, p_value = stats.f_oneway(low_earthquakes["depth"].dropna(), mid_earthquakes["depth"].dropna(), high_earthquakes["depth"].dropna())
print("F-statistic:", f_stat)
print("p-value:", p_value)

if p_value <= 0.05:
    print("Reject H0: at least one region has a significantly different mean depth.")
else:
    print("Fail to reject H0: no significant difference in mean depth across regions.")



We see that between the three regions, there is at least one region with a significantly different mean depth. Let's dig down further and see exactly which region is significantly different using Tukey's HSD test: 

In [ ]:
#because depth and region category are from two different dataframes, we merge them by stacking each column
tukey_df = pd.DataFrame({
    "depth": pd.concat([low_earthquakes["depth"],mid_earthquakes["depth"],high_earthquakes["depth"]], ignore_index=True),
    "region category": (["Low Latitude"] * len(low_earthquakes) + ["Mid Latitude"] * len(mid_earthquakes) + ["High Latitude"] * len(high_earthquakes))
})

tukey = pairwise_tukeyhsd(endog=tukey_df["depth"], groups=tukey_df["region category"], alpha=0.05)

print(tukey)

Here we can see that all three regions have significally different mean depths from the others. This suggests a strong relationship between latitude region and earthquake depth.

Let's plot the data to observe other statistical characteristics:

In [ ]:
data = [
    low_earthquakes["depth"],
    mid_earthquakes["depth"],
    high_earthquakes["depth"]
]

plt.boxplot(data, tick_labels=["Low", "Mid", "High"])
plt.title("Earthquake Depth Comparison")
plt.ylabel("Depth")
plt.show()

The box plot shows that the three latitude regions have different distributions of earthquake depths. The Low Latitude region has the highest median depth, the Mid Latitude region has the lowest median depth, and the High Latitude region is in between those two. These visual differences are consistent with our ANOVA and Tukey HSD results.One thing to note is that there are so many outliers for the earthquake depth.

Now let's do the same analysis for magnitude:

In [ ]:
#print the avg mags
print("Mean magnitude of lower lat regions:",low_earthquakes['mag'].mean())
print("Mean magnitude of middle lat regions:", mid_earthquakes['mag'].mean())
print("Mean magnitude of higher lat regions:", high_earthquakes['mag'].mean())

#perform the ANOVA test
f_stat, p_value = stats.f_oneway(low_earthquakes["mag"].dropna(), mid_earthquakes["mag"].dropna(), high_earthquakes["mag"].dropna())
print("F-statistic:", f_stat)
print("p-value:", p_value)

if p_value <= 0.05:
    print("Reject H0: at least one region has a significantly different mean magnitude.")
else:
    print("Fail to reject H0: no significant difference in mean magnitude across regions.")

Here we see that at least one region has a significantly different mean magnitude. Let's dig down even further and perform Tukey's HSD test for this result as well:

In [ ]:
#because mag and region category are from two different dataframes, we merge our two necessary columns by stacking each column
tukey_df = pd.DataFrame({
    "mag": pd.concat([low_earthquakes["mag"],mid_earthquakes["mag"],high_earthquakes["mag"]], ignore_index=True),
    "region category": (["Low Latitude"] * len(low_earthquakes) + ["Mid Latitude"] * len(mid_earthquakes) + ["High Latitude"] * len(high_earthquakes))
})

tukey = pairwise_tukeyhsd(endog=tukey_df["mag"], groups=tukey_df["region category"], alpha=0.05)

print(tukey)

Here we can see that all three regions have significantly different mean magnitudes from one another. This indicates that earthquake magnitude varies significantly across latitude regions, which is the same conclusion we derived for the earthquake depth.

Let's plot for magnitude:

In [ ]:
data = [
    low_earthquakes["mag"],
    mid_earthquakes["mag"],
    high_earthquakes["mag"]
]

plt.boxplot(data, tick_labels=["Low", "Mid", "High"])
plt.title("Earthquake Magnitude Comparison")
plt.ylabel("Magnitude")
plt.show()

Here we see that the Low Latitude region has the highest median earthquake magnitude, while the Mid Latitude and High Latitude regions have slightly lower and more similar medians. This is still consistent with the Anova and Tukey HSD test results. Notice that there are fewer outliers in the magnitude data as well.